In [ ]:
import json
import os
import re
import sys
from typing import Optional

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

credentials = os.getenv("GIGACHAT_CREDENTIALS") or os.getenv("GIGA_KEY")

llm = GigaChat(
    credentials=credentials,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)


class RentalExtraction(BaseModel):
    count_adults: Optional[int] = Field(default=None)
    count_children: Optional[int] = Field(default=None)
    start_date: Optional[str] = Field(default=None, description="Формат DD.MM")
    nights: Optional[int] = Field(default=None)
    price_per_day: Optional[int] = Field(default=None)
    remarks: Optional[str] = Field(default=None)


parser = PydanticOutputParser(pydantic_object=RentalExtraction)

base_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Извлекай поля count_adults, count_children, start_date, nights, price_per_day, remarks из текста заявки. "
            "Верни только JSON."
        ),
        (
            "human",
            "Текст:\n{text}\n\n{format_instructions}",
        ),
    ]
)

detailed_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Извлекай структуру заявки на аренду. Правила: "
            "1) count_adults и count_children заполняй только если это явно сказано, иначе null. "
            "2) start_date: самая ранняя дата заезда в формате DD.MM, если нет даты -> null. "
            "3) nights: число ночей; если указан диапазон дат, считай разницу между датами; "
            "если есть формулировка 'на X дней', ставь X; если нет данных -> null. "
            "4) price_per_day: желаемая цена в сутки числом, при диапазоне бери максимум, если нет -> null. "
            "5) remarks: особые пожелания (парковка, близость к морю, бассейн, удобства и т.д.), иначе null. "
            "Верни только JSON."
        ),
        (
            "human",
            "Текст:\n{text}\n\n{format_instructions}",
        ),
    ]
)

few_shot_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Извлекай структуру заявки и возвращай только JSON по схеме. Примеры:\n"
            "Текст: 'Двое взрослых и ребенок с 11 по 22 августа, до 3000р/сут, нужен бассейн'\n"
            "JSON: {{\"count_adults\": 2, \"count_children\": 1, \"start_date\": \"11.08\", \"nights\": 11, \"price_per_day\": 3000, \"remarks\": \"нужен бассейн\"}}\n"
            "Текст: 'Ищем двухместный номер с 1.09 по 8.09 рядом с морем'\n"
            "JSON: {{\"count_adults\": null, \"count_children\": null, \"start_date\": \"01.09\", \"nights\": 7, \"price_per_day\": null, \"remarks\": \"рядом с морем\"}}\n"
            "Текст: 'Семья из 5 человек, двое взрослых и трое детей, с 20 по 30 сентября'\n"
            "JSON: {{\"count_adults\": 2, \"count_children\": 3, \"start_date\": \"20.09\", \"nights\": 10, \"price_per_day\": null, \"remarks\": null}}"
        ),
        (
            "human",
            "Текст:\n{text}\n\n{format_instructions}",
        ),
    ]
)

cot_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Сначала скрыто проанализируй текст пошагово, затем верни только финальный JSON. "
            "Пошагово определи: состав проживающих, дату заезда, длительность, цену, пожелания. "
            "Если поля нет - null. Выводи только JSON."
        ),
        (
            "human",
            "Текст:\n{text}\n\n{format_instructions}",
        ),
    ]
)


def safe_parse_json(raw_text: str) -> dict:
    try:
        return json.loads(raw_text)
    except Exception:
        match = re.search(r"\{.*\}", raw_text, flags=re.DOTALL)
        if not match:
            return {
                "count_adults": None,
                "count_children": None,
                "start_date": None,
                "nights": None,
                "price_per_day": None,
                "remarks": None,
            }
        try:
            return json.loads(match.group(0))
        except Exception:
            return {
                "count_adults": None,
                "count_children": None,
                "start_date": None,
                "nights": None,
                "price_per_day": None,
                "remarks": None,
            }


def normalize_start_date(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if not s:
        return None
    m = re.search(r"(\d{1,2})\.(\d{1,2})", s)
    if not m:
        return None
    return f"{int(m.group(1)):02d}.{int(m.group(2)):02d}"


def to_int_or_none(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value)
    m = re.search(r"\d+", s)
    return int(m.group(0)) if m else None


def run_technique(prompt):
    chain = prompt | llm | StrOutputParser()
    rows = []
    for _, row in test_df.iterrows():
        text = row["text"]
        raw = chain.invoke({"text": text, "format_instructions": parser.get_format_instructions()})
        parsed = safe_parse_json(raw)
        rows.append(
            {
                "count_adults": to_int_or_none(parsed.get("count_adults")),
                "count_children": to_int_or_none(parsed.get("count_children")),
                "start_date": normalize_start_date(parsed.get("start_date")),
                "nights": to_int_or_none(parsed.get("nights")),
                "price_per_day": to_int_or_none(parsed.get("price_per_day")),
                "remarks": parsed.get("remarks"),
            }
        )
    return pd.DataFrame(rows)


full_df = pd.read_csv("rental_32.csv", sep=";")
test_df = full_df.head(15).copy()

manual_labels = pd.DataFrame(
    [
        {"count_adults": 2, "count_children": 2, "start_date": "30.06", "nights": 7, "price_per_day": None},
        {"count_adults": 2, "count_children": 2, "start_date": "05.08", "nights": 7, "price_per_day": None},
        {"count_adults": None, "count_children": None, "start_date": "16.08", "nights": 13, "price_per_day": None},
        {"count_adults": None, "count_children": None, "start_date": "01.09", "nights": 7, "price_per_day": None},
        {"count_adults": 2, "count_children": 2, "start_date": "18.08", "nights": 6, "price_per_day": None},
        {"count_adults": None, "count_children": None, "start_date": "07.09", "nights": 7, "price_per_day": 700},
        {"count_adults": 1, "count_children": 1, "start_date": "11.08", "nights": 11, "price_per_day": None},
        {"count_adults": None, "count_children": None, "start_date": "20.06", "nights": 8, "price_per_day": None},
        {"count_adults": 2, "count_children": 1, "start_date": None, "nights": 10, "price_per_day": None},
        {"count_adults": 2, "count_children": 1, "start_date": "17.08", "nights": 10, "price_per_day": None},
        {"count_adults": 3, "count_children": 0, "start_date": "18.08", "nights": 7, "price_per_day": None},
        {"count_adults": None, "count_children": None, "start_date": "19.08", "nights": 10, "price_per_day": None},
        {"count_adults": 2, "count_children": 3, "start_date": "20.09", "nights": 10, "price_per_day": None},
        {"count_adults": None, "count_children": None, "start_date": None, "nights": None, "price_per_day": None},
        {"count_adults": None, "count_children": None, "start_date": "02.07", "nights": 8, "price_per_day": 2000},
    ]
)

techniques = {
    "Базовый": base_prompt,
    "Детальный": detailed_prompt,
    "Few-shot": few_shot_prompt,
    "Chain-of-Thought": cot_prompt,
}
technique_slug = {
    "Базовый": "basic",
    "Детальный": "detailed",
    "Few-shot": "few_shot",
    "Chain-of-Thought": "cot",
}

field_names = ["count_adults", "count_children", "start_date", "nights", "price_per_day"]
metric_fields = ["count_adults", "count_children", "start_date", "nights"]

all_results = {}
score_rows = []

for technique_name, prompt in techniques.items():
    pred_df = run_technique(prompt)
    all_results[technique_name] = pred_df

    merged = test_df[["text"]].reset_index(drop=True).copy()
    for field in field_names:
        merged[f"expected_{field}"] = manual_labels[field]
        merged[f"pred_{field}"] = pred_df[field]

    for field in field_names:
        left = merged[f"expected_{field}"]
        right = merged[f"pred_{field}"]
        merged[f"ok_{field}"] = (left == right) | (left.isna() & right.isna())

    merged.to_csv(
        f"rental_32_{technique_slug[technique_name]}_results.csv",
        index=False,
        encoding="utf-8-sig",
    )

    metric_values = {field: float(merged[f"ok_{field}"].mean()) for field in metric_fields}
    metric_values["avg_1_4"] = sum(metric_values.values()) / len(metric_fields)
    metric_values["technique"] = technique_name
    score_rows.append(metric_values)

scores_df = pd.DataFrame(score_rows).sort_values("avg_1_4", ascending=False)
advanced_scores_df = scores_df[scores_df["technique"].isin(["Детальный", "Few-shot", "Chain-of-Thought"])].copy()
best_technique = advanced_scores_df.iloc[0]["technique"]

print("Оценка точности по полям 1-4:")
print(scores_df[["technique", "count_adults", "count_children", "start_date", "nights", "avg_1_4"]])
print(f"Лучшая продвинутая техника: {best_technique}")

best_df = all_results[best_technique]
print("\nПример результатов лучшей продвинутой техники:")
print(best_df.head(15))

# Тестирование на отдельных примерах
# Тестовые данные различной сложности
test_cases = [
    "Семья из пяти человек ищет большую квартиру",
    "Молодая пара с ребенком рассматривает двухкомнатную квартиру",
    "Двое взрослых и двое детей с 05.08 на 7 дней, бюджет до 4500 в сутки",
    "Нужен номер для троих с 12.07 по 20.07, рядом с морем и с парковкой",
]

# Тестирование всех техник
chains = {
    "Базовый": base_prompt | llm | StrOutputParser(),
    "Детальный": detailed_prompt | llm | StrOutputParser(),
    "Few-shot": few_shot_prompt | llm | StrOutputParser(),
    "Chain-of-Thought": cot_prompt | llm | StrOutputParser(),
}

for text in test_cases:
    print(f"\nТекст: {text}")
    for name, chain in chains.items():
        try:
            result = chain.invoke({"text": text, "format_instructions": parser.get_format_instructions()})
            print(f"{name}: {result}")
        except Exception as e:
            print(f"{name}: Ошибка - {e}")
